# SummaryEvaluation - Meeting Summary Evaluator
**Course:** CSC 603 - Generative AI | **Team:** Adrian Aquino, Charlie Huynh, Will Brust | **Spring 2026**

## Install Libraries

In [ ]:
%pip install -q torch transformers accelerate huggingface_hub python-dotenv ipywidgets

print('Done!')

## Import Libraries

In [ ]:
import os
import json
import torch
from transformers import pipeline
from huggingface_hub import login
from dotenv import load_dotenv, set_key

# only import colab files if running on Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Done!')

## Hugging Face Login

In [ ]:
if IN_COLAB:
    # on Colab: load from Colab Secrets (Secrets tab in left sidebar)
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    # locally: load from .env file
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

    if not HF_TOKEN:
        print('Enter your Hugging Face token when prompted, then press Enter.')
        HF_TOKEN = input('HF Token: ')
        set_key('.env', 'HF_TOKEN', HF_TOKEN)
        print('Token saved to .env')

login(token=HF_TOKEN)
print('Logged in!')

## Load the Model

In [ ]:
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

# use GPU if available, otherwise CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

llm = pipeline(
    task='text-generation',
    model=MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
)

print('Model loaded!')

## Summarize Function

In [ ]:
ROLE = "You are an AI assistant that summarizes meeting transcripts into structured JSON."

TASK = """Summarize the following meeting transcript.
Return ONLY a JSON object with exactly these 4 keys:
- "summary": a short 2-3 sentence overview of the meeting
- "decisions": a list of decisions that were made
- "assigned_tasks": a list of objects with "who", "what", and "due" fields
- "open_questions": a list of questions that were raised but not resolved

Do not include any explanation or text outside the JSON."""


def summarize_transcript(transcript, max_tokens=1024):
    """Sends a transcript to the model and returns a parsed JSON summary.

    Args:
        transcript (str): The meeting transcript text.
        max_tokens (int): Max tokens to generate.

    Returns:
        dict: JSON summary with summary, decisions, assigned_tasks, open_questions.
    """
    messages = [
        {'role': 'system', 'content': ROLE},
        {'role': 'user',   'content': f'{TASK}\n\nTranscript:\n{transcript}'}
    ]
    response = llm(messages, max_new_tokens=max_tokens, do_sample=False, temperature=1.0)
    raw      = response[0]['generated_text'][-1]['content']

    try:
        start  = raw.index('{')
        end    = raw.rindex('}') + 1
        return json.loads(raw[start:end])
    except (ValueError, json.JSONDecodeError):
        return {'summary': raw, 'decisions': [], 'assigned_tasks': [], 'open_questions': []}


print('Function defined!')

## Load Transcript Files
Scans for all `GeneratedMockTranscript-XX.txt` files generated by RecapAI.

In [ ]:
# scan for all GeneratedMockTranscript-XX.txt files
transcript_files = sorted([
    f for f in os.listdir('.')
    if f.startswith('GeneratedMockTranscript-') and f.endswith('.txt')
])

if transcript_files:
    print(f'Found {len(transcript_files)} transcript files:')
    for f in transcript_files:
        print(f'  {f}')
else:
    print('No GeneratedMockTranscript-XX.txt files found.')
    print('Run option 3 in RecapAI first to generate transcripts.')

## Load Real-World Transcripts
Supports QMSum and TCR JSON formats. Place `.json` files in this folder to evaluate against real meetings.

In [ ]:
# QMSum format: {"meeting_transcripts": [{"speaker": "...", "content": "..."}]}
def load_qmsum(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    turns = data.get('meeting_transcripts', [])
    return '\n'.join(f"{t['speaker']}: {t['content']}" for t in turns)


# TCR format: nested data_source > meeting_name > topics > transcripts
def load_tcr(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    lines = []
    for source in data.values():
        for meeting in source.values():
            for topic in meeting.get('topics', {}).values():
                for turn in topic.get('transcripts', []):
                    lines.append(f"{turn.get('speaker', 'Speaker')}: {turn.get('contents', '')}")
    return '\n'.join(lines)


# auto-detect which format the file is and load it
def detect_and_load(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    if 'meeting_transcripts' in data:
        return load_qmsum(filepath)
    return load_tcr(filepath)


# scan for real-world JSON files, skip evaluation outputs and output.json
realworld_files = sorted([
    f for f in os.listdir('.')
    if f.endswith('.json')
    and not f.endswith('_evaluation.json')
    and f != 'output.json'
])

if realworld_files:
    print(f'Found {len(realworld_files)} real-world transcript file(s):')
    for f in realworld_files:
        print(f'  {f}')
else:
    print('No real-world transcript JSON files found.')
    print('Download QMSum or TCR .json files and place them here to test against real meetings.')

print('Functions defined!')

## Evaluation Prompt and Functions

In [ ]:
EVAL_PROMPT = """Score this AI meeting summary against the original transcript.

Return ONLY valid JSON with these keys:
- scores: summary (1-5), decisions (1-5), assigned_tasks (1-5), open_questions (1-5), overall (1-10)
- issues: list of errors found
- prompt_suggestions: list of fixes to improve the prompt

Scores: 5 = correct and complete, 1 = wrong or missing

Watch for:
- Tasks assigned to the wrong speaker
- Facts or statements listed as decisions
- Resolved items listed as open questions
- Deadlines not mentioned in the transcript
- Duplicate or missing items

For suggestions, be direct. Example:
Add to prompt: Only assign a task to someone who explicitly volunteered.
"""


def evaluate(transcript, summary):
    """Sends transcript and summary to the model and returns a scored evaluation.

    Args:
        transcript (str): The original meeting transcript.
        summary (dict): The JSON summary output from RecapAI.

    Returns:
        dict: Scores, issues, and prompt suggestions.
    """
    messages = [
        {'role': 'system', 'content': EVAL_PROMPT},
        {'role': 'user',   'content': f'Transcript:\n{transcript}\n\nSummary:\n{json.dumps(summary, indent=2)}'}
    ]

    print('Sending to model for evaluation...')
    response = llm(messages, max_new_tokens=1024, do_sample=False, temperature=1.0)
    raw = response[0]['generated_text'][-1]['content']
    print('Response received. Parsing...')

    return try_parse_eval(raw)


def try_parse_eval(raw):
    """Parses evaluation output as JSON with a fallback to raw text.

    Args:
        raw (str): Raw text from the model.

    Returns:
        dict: Parsed evaluation or fallback with raw text.
    """
    try:
        start  = raw.index('{')
        end    = raw.rindex('}') + 1
        parsed = json.loads(raw[start:end])
        print('Parsed successfully.')
        return parsed
    except (ValueError, json.JSONDecodeError):
        print('Could not parse output. Returning raw text.')
        return {'raw': raw}


def display_results(result):
    """Prints scores, issues, and prompt suggestions in a readable format.

    Args:
        result (dict): Parsed evaluation output from evaluate().
    """
    if 'raw' in result:
        print('Raw output (could not parse):')
        print(result['raw'])
        return

    scores = result.get('scores', {})
    print('=' * 40)
    print('EVALUATION SCORES')
    print('=' * 40)
    print('  Summary:        ' + str(scores.get('summary', 'N/A')) + ' / 5')
    print('  Decisions:      ' + str(scores.get('decisions', 'N/A')) + ' / 5')
    print('  Assigned Tasks: ' + str(scores.get('assigned_tasks', 'N/A')) + ' / 5')
    print('  Open Questions: ' + str(scores.get('open_questions', 'N/A')) + ' / 5')
    print('  Overall:        ' + str(scores.get('overall', 'N/A')) + ' / 10')
    print()

    issues = result.get('issues', [])
    if issues:
        print('ISSUES FOUND')
        print('-' * 40)
        for i, issue in enumerate(issues, 1):
            print('  ' + str(i) + '. ' + issue)
        print()

    suggestions = result.get('prompt_suggestions', [])
    if suggestions:
        print('PROMPT IMPROVEMENT SUGGESTIONS')
        print('-' * 40)
        for i, s in enumerate(suggestions, 1):
            print('  ' + str(i) + '. ' + s)
        print()


print('Functions defined!')

## Run — Evaluate All Transcripts

In [ ]:
# build combined list: (label, transcript_text) from mock and real-world sources
all_transcripts = []

for filename in transcript_files:
    with open(filename, 'r') as f:
        all_transcripts.append((filename, f.read()))

for filename in realworld_files:
    try:
        all_transcripts.append((filename, detect_and_load(filename)))
    except Exception as e:
        print(f'Skipping {filename}: {e}')

if not all_transcripts:
    print('No transcripts found.')
    print('Run RecapAI option 3 to generate mock transcripts, or add QMSum/TCR .json files here.')
else:
    all_scores = []

    for label, transcript in all_transcripts:
        print(f'\n{"=" * 40}')
        print(f'Processing: {label}')
        print('=' * 40)

        print('Summarizing...')
        summary = summarize_transcript(transcript)

        result = evaluate(transcript, summary)
        display_results(result)

        out_name = label.replace('.txt', '').replace('.json', '') + '_evaluation.json'
        with open(out_name, 'w') as f:
            json.dump({'file': label, 'summary': summary, 'evaluation': result}, f, indent=2)
        print(f'Saved: {out_name}')

        if 'scores' in result:
            all_scores.append({'file': label, **result['scores']})

    if all_scores:
        print('\n' + '=' * 50)
        print('OVERALL RESULTS')
        print('=' * 50)
        print(f"{'File':<35} {'Sum':>4} {'Dec':>4} {'Tasks':>6} {'OQ':>4} {'Total':>6}")
        print('-' * 50)
        for s in all_scores:
            print(f"{s['file']:<35} {str(s.get('summary','?')):>4} {str(s.get('decisions','?')):>4} {str(s.get('assigned_tasks','?')):>6} {str(s.get('open_questions','?')):>4} {str(s.get('overall','?')):>6}")

    print(f'\nDone! Evaluated {len(all_transcripts)} transcript(s).')

## Generate Improved Prompts
Reads all evaluation results and generates updated ROLE and TASK prompts you can copy into RecapAI.

In [ ]:
# collect all prompt suggestions from saved evaluation files
all_suggestions = []
eval_files = sorted([f for f in os.listdir('.') if f.endswith('_evaluation.json')])

for filename in eval_files:
    with open(filename, 'r') as f:
        data = json.load(f)
    suggestions = data.get('evaluation', {}).get('prompt_suggestions', [])
    all_suggestions.extend(suggestions)

if not all_suggestions:
    print('No suggestions found. Run the evaluation cells first.')
else:
    print(f'Collected {len(all_suggestions)} suggestion(s) from {len(eval_files)} evaluation file(s).')

    suggestions_text = '\n'.join(f'- {s}' for s in all_suggestions)

    improve_prompt = (
        'You are improving an AI prompt based on evaluation feedback.\n\n'
        'Current ROLE:\n' + ROLE + '\n\n'
        'Current TASK:\n' + TASK + '\n\n'
        'Suggestions from evaluation:\n' + suggestions_text + '\n\n'
        'Return only two clearly labeled blocks:\n'
        'ROLE: <improved role>\n'
        'TASK: <improved task>\n\n'
        'Do not add any other text.'
    )

    messages = [
        {'role': 'system', 'content': 'You improve AI prompts based on evaluation feedback.'},
        {'role': 'user',   'content': improve_prompt}
    ]

    print('Generating improved prompts...')
    response = llm(messages, max_new_tokens=512, do_sample=False, temperature=1.0)
    raw = response[0]['generated_text'][-1]['content']

    print('\n' + '=' * 50)
    print('GENERATED PROMPTS:')
    print('=' * 50)
    print(raw)

    # parse new ROLE and TASK from the model output
    new_role = None
    new_task = None
    lines = raw.strip().split('\n')
    for i, line in enumerate(lines):
        if line.startswith('ROLE:') and new_role is None:
            new_role = line[5:].strip()
        elif line.startswith('TASK:') and new_task is None:
            task_lines = [line[5:].strip()] + lines[i + 1:]
            new_task = '\n'.join(task_lines).strip()
            break

    if not new_role or not new_task:
        print('\nCould not parse ROLE or TASK. Copy manually from above.')
    else:
        print('\nReplace the prompts in RecapAI.ipynb with the new ones?')
        choice = input('Enter yes or no: ').strip().lower()

        if choice == 'yes':
            with open('RecapAI.ipynb', 'r', encoding='utf-8') as f:
                nb = json.load(f)

            for cell in nb['cells']:
                if cell.get('id') == 'cell-12':
                    source = ''.join(cell['source'])

                    # replace ROLE line
                    role_start = source.index('ROLE = "')
                    role_end   = source.index('"', role_start + 8) + 1
                    source     = source[:role_start] + f'ROLE = "{new_role}"' + source[role_end:]

                    # replace TASK triple-quoted block
                    task_start = source.index('TASK = """')
                    task_end   = source.index('"""', task_start + 10) + 3
                    source     = source[:task_start] + f'TASK = """{new_task}"""' + source[task_end:]

                    cell['source'] = source.splitlines(True)
                    break

            with open('RecapAI.ipynb', 'w', encoding='utf-8') as f:
                json.dump(nb, f, indent=1, ensure_ascii=False)

            print('Done! RecapAI.ipynb has been updated with the new prompts.')
        else:
            print('Cancelled. Copy the prompts manually from above if needed.')